# Bankruptcy predictions

Load an exported model, score new companies, and save review flags. No retraining occurs here.

1. Copy `exported_models/` and your unseen-company CSV into the project folder.
2. Run imports, configure settings, and see available exported models.
3. Fill the empty filename in `MODEL_PATH`, check `INPUT_PATH`, then rerun Settings and all following cells.

Use the same financial features, definitions, units, and normalization as training. For local setup, see the README.


## Obtain the models

Run "Export Models" section after model development in the training notebook, which will save all 13 named configurations to `exported_models/`, with `model_index.csv` listing the files.

Each bundle contains the fitted pipeline, feature names, threshold, costs, and package versions. Existing fits are preserved; CV-only configurations are fitted once on training data. Grid-search trials and fold models are not exported.

For Colab, download and extract the export folder locally. Load only trusted bundles.


In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import csv
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted


## Settings

Choose a model filename and input CSV. Paths use the working directory printed below. Set `ID_COLUMNS` to retain company identifiers.


In [ ]:
# Paths are relative to the notebook kernel's working directory
EXPORT_DIR = Path("exported_models")

# Example selection only: choose any filename shown in Available exported models
# MODEL_PATH = EXPORT_DIR / "11_cost_sensitive_decision_tree.joblib"
MODEL_PATH = EXPORT_DIR / ""
INPUT_PATH = Path("unseen_companies.csv")
# Separate output for each model, so switching models preserves earlier results.
OUTPUT_PATH = Path("predictions") / f"{MODEL_PATH.stem}_predictions.csv"

# Optional identifier columns in the input CSV (not model features).
# Example: ["Company ID", "Company Name"]
ID_COLUMNS = []

# None preserves this model's saved threshold or default prediction rule.
# Only supply a value already selected using training/validation data.
THRESHOLD_OVERRIDE = None

# False prevents accidentally replacing a previous prediction export.
OVERWRITE_OUTPUT = False

print("Working directory:", Path.cwd())
print("Selected model file:", MODEL_PATH)
print("Prediction output:", OUTPUT_PATH)


## Available exported models

Lists all bundles in `EXPORT_DIR`, with names from the optional CSV index. Choose any filename in `MODEL_PATH`.


In [ ]:
available_files = sorted(path for path in EXPORT_DIR.glob("*.joblib") if path.is_file())
if not available_files:
    raise FileNotFoundError(
        f"No exported model bundles found in {EXPORT_DIR.resolve()}. "
        "Run the training notebook's Export models cell and copy its export folder here."
    )

# Resolve index entries by filename so a copied folder remains portable.
index_path = EXPORT_DIR / "model_index.csv"
model_names = {}
if index_path.is_file():
    model_index = pd.read_csv(index_path)
    if {"Model", "File"} <= set(model_index.columns):
        model_names = {
            Path(str(row["File"]).replace("\\", "/")).name: row["Model"]
            for _, row in model_index.iterrows()
        }

available_models_df = pd.DataFrame([
    {"Filename": path.name, "Model": model_names.get(path.name, "Not listed in index")}
    for path in available_files
])
display(available_models_df)
print("Choose any Filename above in MODEL_PATH, then rerun Settings and subsequent cells.")


## 1. Load the model

Load the selected bundle and check package versions and features. Labels are 0 = not bankrupt and 1 = bankrupt.

The saved threshold is used unless overridden. `None` uses the model's default rule; a numeric threshold flags scores at or above it.


In [ ]:
def validate_threshold(value):
    if value is None:
        return None
    if isinstance(value, (bool, str)) or not np.isscalar(value):
        raise ValueError("Threshold must be None or a number between 0 and 1.")
    value = float(value)
    if not np.isfinite(value) or not 0 <= value <= 1:
        raise ValueError("Threshold must be between 0 and 1.")
    return value


def load_model_bundle(path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Model bundle not found: {path.resolve()}. "
            "Run the training notebook\'s Export models cell, copy exported_models/ locally, "
            "and choose an existing bundle in MODEL_PATH."
        )
    bundle = joblib.load(path)
    if not isinstance(bundle, dict) or not {"model", "feature_names"} <= bundle.keys():
        raise ValueError("Expected a bundle containing 'model' and 'feature_names'.")

    mismatches = []
    for package, expected in bundle.get("package_versions", {}).items():
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = "not installed"
        if installed != expected:
            mismatches.append(f"{package}: saved={expected}, installed={installed}")
    if mismatches:
        raise RuntimeError("Use the training package versions:\n" + "\n".join(mismatches))
    if not bundle.get("package_versions"):
        warnings.warn("No package version metadata: compatibility has not been verified.")

    model = bundle["model"]
    check_is_fitted(model)
    if not callable(getattr(model, "predict_proba", None)):
        raise ValueError("The saved model must support predict_proba.")

    features = list(bundle["feature_names"])
    if not features or not all(isinstance(name, str) and name.strip() for name in features):
        raise ValueError("feature_names must contain nonempty column names.")
    canonical = [name.strip() for name in features]
    if len(set(canonical)) != len(canonical):
        raise ValueError("Saved feature names are ambiguous after trimming whitespace.")
    if "Bankrupt?" in canonical:
        raise ValueError("The target Bankrupt? must not be an input feature.")
    if hasattr(model, "n_features_in_") and model.n_features_in_ != len(features):
        raise ValueError("Saved feature count does not match the fitted model.")
    if hasattr(model, "feature_names_in_") and list(model.feature_names_in_) != features:
        raise ValueError("Saved feature names/order do not match the fitted model.")

    classes = list(model.classes_)
    if len(classes) != 2 or set(classes) != {0, 1} or bundle.get("positive_class", 1) != 1:
        raise ValueError("Expected labels 0 = not bankrupt and 1 = bankrupt.")
    validate_threshold(bundle.get("threshold"))
    return bundle


In [ ]:
bundle = load_model_bundle(MODEL_PATH)
model = bundle["model"]
feature_names = list(bundle["feature_names"])
threshold = validate_threshold(
    THRESHOLD_OVERRIDE if THRESHOLD_OVERRIDE is not None
    else bundle.get("threshold")
)

print("Selected model:", bundle.get("model_name", type(model).__name__))
print("Required features:", len(feature_names))
print("Decision rule:", "model.predict()" if threshold is None else f"score >= {threshold}")


## 2. Prepare unseen data

Match feature names and order, convert values to numbers, and reject invalid inputs. Extra columns are excluded; rows and selected IDs are preserved.

Missing values require a saved fitted imputer. Scaling and imputation reuse training settings.


In [ ]:
def read_unseen_csv(path):
    if not path.is_file():
        raise FileNotFoundError(f"Unseen-company CSV not found: {path.resolve()}")
    # Check raw headers before pandas can rename duplicate columns.
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        header = next(csv.reader(handle), [])
    normalized = [name.strip() for name in header]
    if not normalized or any(not name for name in normalized):
        raise ValueError("CSV headers must be nonempty.")
    if len(set(normalized)) != len(normalized):
        raise ValueError("CSV contains duplicate headers after trimming whitespace.")
    frame = pd.read_csv(path, dtype="string", encoding="utf-8-sig")
    frame.columns = normalized
    if frame.empty:
        raise ValueError("The CSV contains no company rows.")
    return frame


def prepare_features(frame, features, fitted_model):
    canonical = [name.strip() for name in features]
    missing = [name for name in canonical if name not in frame.columns]
    if missing:
        raise ValueError(f"Missing required feature columns: {missing}")

    numeric = pd.DataFrame(index=frame.index)
    invalid = []
    for original, normalized in zip(features, canonical):
        values = frame[normalized].astype("string").str.strip().replace("", pd.NA)
        converted = pd.to_numeric(values, errors="coerce")
        if (values.notna() & converted.isna()).any():
            invalid.append(normalized)
        numeric[original] = converted.to_numpy(dtype=float, na_value=np.nan)
    if invalid:
        raise ValueError(
            f"Nonnumeric feature values in: {invalid}. "
            "Provide numbers in training units; percent signs/currency are not auto-converted."
        )
    if np.isinf(numeric.to_numpy()).any():
        raise ValueError("Features contain infinity. Correct the underlying financial ratios.")

    missing_counts = numeric.isna().sum()
    if missing_counts.any():
        parameters = fitted_model.get_params(deep=True).values()
        has_imputer = any(
            isinstance(step, SimpleImputer) and hasattr(step, "statistics_")
            for step in parameters
        )
        if not has_imputer:
            affected = missing_counts[missing_counts > 0].to_dict()
            raise ValueError(
                f"Missing feature values: {affected}. "
                "This model has no fitted SimpleImputer; provide complete inputs."
            )
    return numeric


In [ ]:
companies = read_unseen_csv(INPUT_PATH)

id_columns = [name.strip() for name in ID_COLUMNS]
if len(id_columns) != len(set(id_columns)):
    raise ValueError("ID_COLUMNS contains duplicates.")
missing_ids = [name for name in id_columns if name not in companies.columns]
if missing_ids:
    raise ValueError(f"Requested identifier columns are missing: {missing_ids}")
if "Bankrupt?" in id_columns:
    raise ValueError("Bankrupt? is the target, not a company identifier.")

X_new = prepare_features(companies, feature_names, model)

excluded = [
    name for name in companies.columns
    if name not in {feature.strip() for feature in feature_names}
]
print("Company rows:", len(X_new))
print("Non-feature columns excluded from model:", excluded)
print("Missing feature values handled by saved imputer:", int(X_new.isna().sum().sum()))
display(X_new.head())


## 3. Predict

Generate bankruptcy scores and review flags using the saved decision rule. Unlabeled data cannot measure accuracy or realised cost.


In [ ]:
def predict_companies(fitted_model, features, decision_threshold):
    decision_threshold = validate_threshold(decision_threshold)
    positive_column = list(fitted_model.classes_).index(1)
    scores = np.asarray(fitted_model.predict_proba(features))[:, positive_column]
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Model returned invalid probability scores.")
    if decision_threshold is None:
        predictions = np.asarray(fitted_model.predict(features), dtype=int)
    else:
        predictions = (scores >= decision_threshold).astype(int)
    return scores, predictions


In [ ]:
risk_scores, predictions = predict_companies(model, X_new, threshold)

results = pd.DataFrame(index=companies.index)
# One-based company record number, excluding the CSV header.
results["Input row"] = np.arange(1, len(companies) + 1)
for name in id_columns:
    # Prefix avoids collisions with prediction column names.
    results[f"ID: {name}"] = companies[name]

results["Bankruptcy risk score"] = risk_scores
results["Predicted Bankrupt?"] = predictions
results["Flag for further review"] = predictions == 1
results["Model"] = bundle.get("model_name", type(model).__name__)

display(results.head(20))
print(f"Flagged {int((predictions == 1).sum())} of {len(results)} companies.")


## 4. Save predictions

Write a separate CSV per model under `predictions/`, preserving row order and selected IDs.

To replace an existing result, set `OVERWRITE_OUTPUT = True`; otherwise choose a new `OUTPUT_PATH`.


In [ ]:
if OUTPUT_PATH.resolve() in {INPUT_PATH.resolve(), MODEL_PATH.resolve()}:
    raise ValueError("The output path must differ from the input CSV and model bundle.")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(
    OUTPUT_PATH, index=False, encoding="utf-8-sig",
    mode="w" if OVERWRITE_OUTPUT else "x"
)
print("Predictions saved to:", OUTPUT_PATH.resolve())
